In [0]:
# This file is a Databricks notebook — import via: Workspace → Import → File
# Format: Python (.py) with # COMMAND ---------- as cell separator

%md
# Bronze → Silver: PySpark Transformation

# Reads NDJSON batches from **GCS Bronze**, applies cleaning, deduplication,
# and writes a **Delta Lake** table to **GCS Silver**.

# | Layer  | Format     | Location                        |
# |--------|------------|---------------------------------|
# | Bronze | NDJSON     | `gs://<bucket>/<entity>/year=../` |
# | Silver | Delta Lake | `gs://<bucket>/<entity>/`       |

# > **Serverless Note:** This notebook runs entirely on the **driver** using
# > the Python `google-cloud-storage` and `deltalake` libraries.
# > No Spark workers are involved in reading or writing — this sidesteps
# > all Databricks Serverless filesystem restrictions (DBFS disabled,
# > `/Workspace` read-only for workers, `sparkContext` unavailable).

## Cell 1: Install Dependencies

**Must run first** — `%pip install` restarts the Python interpreter.

In [0]:
%pip install google-cloud-storage google-auth deltalake pyarrow --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.69.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 2: Load Credentials from Databricks Secrets

In [0]:
import json
import io
import os
import shutil
import logging
from concurrent.futures import ThreadPoolExecutor

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# ── Read all secrets from Databricks Secret Scope (provisioned via Terraform) ─
gcp_sa_json    = dbutils.secrets.get(scope="gcp_secrets", key="gcp_sa_key")
gcp_project_id = dbutils.secrets.get(scope="gcp_secrets", key="gcp_project_id")
gcs_bronze     = dbutils.secrets.get(scope="gcp_secrets", key="gcs_bronze_bucket")
gcs_silver     = dbutils.secrets.get(scope="gcp_secrets", key="gcs_silver_bucket")

print(f"[OK] Credentials loaded.")
print(f"     Project : {gcp_project_id}")
print(f"     Bronze  : gs://{gcs_bronze}")
print(f"     Silver  : gs://{gcs_silver}")

[OK] Credentials loaded.
     Project : [REDACTED]
     Bronze  : gs://[REDACTED]
     Silver  : gs://[REDACTED]


## Cell 3: Build GCS Client & Verify Connectivity

In [0]:
from google.oauth2 import service_account
from google.cloud import storage as gcs_storage

sa_info     = json.loads(gcp_sa_json)
credentials = service_account.Credentials.from_service_account_info(sa_info)
gcs_client  = gcs_storage.Client(project=gcp_project_id, credentials=credentials)

try:
    blobs = list(gcs_client.bucket(gcs_bronze).list_blobs(max_results=5))
    print(f"[OK] GCS connected. Sample Bronze files:")
    for blob in blobs:
        print(f"     gs://{gcs_bronze}/{blob.name}")
except Exception as e:
    print(f"[ERROR] {e}")
    raise

[OK] GCS connected. Sample Bronze files:
     gs://[REDACTED]/customer/year=2026/month=05/day=17/batch_00113d9757d3.json
     gs://[REDACTED]/customer/year=2026/month=05/day=17/batch_01e3b39044de.json
     gs://[REDACTED]/customer/year=2026/month=05/day=17/batch_0889c80a1bed.json
     gs://[REDACTED]/customer/year=2026/month=05/day=17/batch_096f37e6d0b4.json
     gs://[REDACTED]/customer/year=2026/month=05/day=17/batch_0e6da0f56599.json


## Cell 4: Define Entity Configuration

In [0]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class EntityConfig:
    name:        str
    primary_key: str
    date_cols:   List[str] = field(default_factory=list)

ENTITIES: List[EntityConfig] = [
    EntityConfig("order",            "order_id",                    ["order_purchase_timestamp", "order_delivered_customer_date", "order_approved_at", "order_delivered_carrier_date", "order_estimated_delivery_date"]),
    EntityConfig("order_item",       "order_item_id",               []),
    EntityConfig("payment",          "order_id",                    []),
    EntityConfig("review",           "review_id",                   ["review_creation_date", "review_answer_timestamp"]),
    EntityConfig("customer",         "customer_id",                 []),
    EntityConfig("product",          "product_id",                  []),
    EntityConfig("product_category", "product_category_name",       []),
    EntityConfig("seller",           "seller_id",                   []),
    EntityConfig("geolocation",      "geolocation_zip_code_prefix", []),
]

print(f"[OK] {len(ENTITIES)} entities registered.")

[OK] 9 entities registered.


## Cell 5: Define Transformation Functions

**Architecture**: All processing runs on the driver using pandas + deltalake.
No Spark workers involved — 100% compatible with Databricks Serverless.

In [0]:
import pandas as pd
from deltalake.writer import write_deltalake

# ── I/O Helpers ───────────────────────────────────────────────────────────────

def _download_blob(blob) -> List[str]:
    """Download one GCS blob, return its non-empty NDJSON lines."""
    content = blob.download_as_text(encoding="utf-8")
    return [l.strip() for l in content.splitlines() if l.strip()]


def read_entity_as_pandas(entity_name: str) -> pd.DataFrame:
    """
    Download all NDJSON files for one entity from GCS Bronze.
    Returns a flat pandas DataFrame with payload + metadata fields.
    Runs entirely on the driver — no Spark, no DBFS, no /Workspace writes.
    """
    blobs = list(gcs_client.bucket(gcs_bronze).list_blobs(prefix=f"{entity_name}/"))
    if not blobs:
        raise FileNotFoundError(f"No files at gs://{gcs_bronze}/{entity_name}/")

    print(f"  [READ]  {len(blobs)} files found. Downloading in parallel...")

    all_lines: List[str] = []
    with ThreadPoolExecutor(max_workers=10) as executor:
        for lines in executor.map(_download_blob, blobs):
            all_lines.extend(lines)

    print(f"  [READ]  {len(all_lines):,} NDJSON lines downloaded.")

    # Parse NDJSON → pandas
    raw_df = pd.read_json(io.StringIO("\n".join(all_lines)), lines=True)

    # Unpack EventEnvelope: {metadata: {...}, payload: {...}}
    if "payload" in raw_df.columns and "metadata" in raw_df.columns:
        payload_df  = pd.json_normalize(raw_df["payload"])
        metadata_df = pd.json_normalize(raw_df["metadata"])
        return pd.concat([payload_df, metadata_df], axis=1)
    elif "payload" in raw_df.columns:
        return pd.json_normalize(raw_df["payload"])
    else:
        return raw_df


def upload_local_dir_to_gcs(local_dir: str, gcs_bucket_name: str, gcs_prefix: str) -> int:
    """
    Upload all files under a local directory to GCS using parallel threads.
    Python's open() on /tmp works fine on the Databricks driver node.
    """
    tasks = []
    for root, _, files in os.walk(local_dir):
        for filename in files:
            local_file = os.path.join(root, filename)
            rel_path   = os.path.relpath(local_file, local_dir).replace("\\", "/")
            blob_name  = f"{gcs_prefix}/{rel_path}"
            tasks.append((local_file, blob_name))

    def _upload(args):
        local_path, blob_name = args
        with open(local_path, "rb") as fh:
            gcs_client.bucket(gcs_bucket_name).blob(blob_name).upload_from_file(fh)
        return 1

    print(f"  [UPLOAD] Uploading {len(tasks)} Delta files in parallel...")
    with ThreadPoolExecutor(max_workers=10) as executor:
        uploaded = sum(executor.map(_upload, tasks))

    print(f"  [UPLOAD] {uploaded} files → gs://{gcs_bucket_name}/{gcs_prefix}/")
    return uploaded


# ── Core Transformation ───────────────────────────────────────────────────────

def process_entity(cfg: EntityConfig) -> dict:
    """
    Bronze → Silver pipeline for one entity (fully driver-side, no Spark workers):

    1. Read   — Download NDJSON from GCS Bronze in parallel
    2. Unpack — Flatten EventEnvelope {metadata, payload} → flat DataFrame
    3. Cast   — Convert date strings → datetime (errors='coerce' → NaT)
    4. Dedup  — Keep latest record per primary_key
    5. Write  — Save as Delta Lake to /tmp (driver local fs)
    6. Upload — Transfer Delta files from /tmp → GCS Silver
    """
    local_delta = f"/tmp/silver/{cfg.name}"
    SEPARATOR   = "=" * 60

    print(f"\n{SEPARATOR}")
    print(f"Entity  : {cfg.name.upper()}")
    print(f"Bronze  : gs://{gcs_bronze}/{cfg.name}/")
    print(f"Silver  : gs://{gcs_silver}/{cfg.name}/")

    # 1 + 2 — Read & Unpack
    df        = read_entity_as_pandas(cfg.name)
    raw_count = len(df)
    assert raw_count > 0, f"[DQ FAIL] No records in Bronze for '{cfg.name}'"
    print(f"  [SHAPE] {raw_count:,} rows × {len(df.columns)} columns")

    # 3 — Type casting (errors='coerce': invalid/empty → NaT, never crashes)
    for col_name in cfg.date_cols:
        if col_name in df.columns:
            df[col_name] = pd.to_datetime(df[col_name], errors="coerce", utc=True)

    df["_silver_loaded_at"] = pd.Timestamp.now(tz="UTC")

    # 4 — Deduplicate: keep latest row per primary key
    pk = cfg.primary_key
    if pk in df.columns:
        order_col = "ingested_at" if "ingested_at" in df.columns else "_silver_loaded_at"
        df = (df
              .sort_values(order_col, ascending=False)
              .drop_duplicates(subset=[pk])
              .reset_index(drop=True))

    silver_count = len(df)
    duplicates   = raw_count - silver_count
    print(f"  [DEDUP] {raw_count:,} raw → {silver_count:,} unique ({duplicates:,} duplicates dropped)")

    # 5 — Write Delta Lake to driver /tmp (no Spark workers, no filesystem restrictions)
    if os.path.exists(local_delta):
        shutil.rmtree(local_delta)

    # Convert to PyArrow Table explicitly — required by deltalake >= 0.10
    # Sanitize object columns: replace Python None/nan with proper nulls
    import pyarrow as pa
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].where(df[col].notna(), other=None)
    arrow_table = pa.Table.from_pandas(df, preserve_index=False)
    write_deltalake(local_delta, arrow_table, mode="overwrite")

    delta_files = sum(len(files) for _, _, files in os.walk(local_delta))
    print(f"  [DELTA] Written to /tmp: {local_delta} ({delta_files} files)")

    # 6 — Upload /tmp Delta → GCS Silver
    uploaded = upload_local_dir_to_gcs(local_delta, gcs_silver, cfg.name)

    return {
        "entity":             cfg.name,
        "raw_count":          raw_count,
        "silver_count":       silver_count,
        "duplicates_dropped": duplicates,
        "silver_path":        f"gs://{gcs_silver}/{cfg.name}/",
        "files_uploaded":     uploaded,
        "status":             "SUCCESS",
    }


print("[OK] All functions defined. Ready to run Cell 6.")

[OK] All functions defined. Ready to run Cell 6.


## Cell 6: Run Pipeline — One Entity at a Time

Each entity is processed and **immediately reported** before moving to the next.
If one entity fails, the pipeline continues with the rest.

In [0]:
import time

results = []
errors  = []

for cfg in ENTITIES:
    start = time.time()
    try:
        result = process_entity(cfg)
        elapsed = time.time() - start
        result["elapsed_sec"] = round(elapsed, 1)
        results.append(result)

        # ── Immediate success report ──────────────────────────────────────────
        print(f"  ✅ SUCCESS  | {cfg.name:<20} | "
              f"{result['silver_count']:>6,} rows | "
              f"{result['files_uploaded']:>3} files | "
              f"{elapsed:.1f}s")

    except AssertionError as dq_err:
        elapsed = time.time() - start
        print(f"  ⚠️  DQ FAIL  | {cfg.name:<20} | {dq_err}")
        errors.append({"entity": cfg.name, "error": str(dq_err), "status": "DQ_FAIL", "elapsed_sec": round(elapsed, 1)})
    except Exception as e:
        elapsed = time.time() - start
        print(f"  ❌ ERROR    | {cfg.name:<20} | {e}")
        errors.append({"entity": cfg.name, "error": str(e), "status": "ERROR", "elapsed_sec": round(elapsed, 1)})

print("\n" + "=" * 60)
print(f"DONE  ✅ {len(results)} succeeded  ❌ {len(errors)} failed  (total: {len(ENTITIES)} entities)")
print("=" * 60)

if results:
    display(spark.createDataFrame(results))

if not errors:
    print("\n✅ All entities processed successfully!")
else:
    for e in errors:
        print(f"  ✗ {e['entity']}: {e['error']}")


Entity  : ORDER
Bronze  : gs://[REDACTED]/order/
Silver  : gs://[REDACTED]/order/
  [READ]  100 files found. Downloading in parallel...
  [READ]  99,441 NDJSON lines downloaded.
  [SHAPE] 99,441 rows × 19 columns
  [DEDUP] 99,441 raw → 99,441 unique (0 duplicates dropped)
  [DELTA] Written to /tmp: /tmp/silver/order (2 files)
  [UPLOAD] Uploading 2 Delta files in parallel...
  [UPLOAD] 2 files → gs://[REDACTED]/order/
  ✅ SUCCESS  | order                | 99,441 rows |   2 files | 30.7s

Entity  : ORDER_ITEM
Bronze  : gs://[REDACTED]/order_item/
Silver  : gs://[REDACTED]/order_item/
  [READ]  113 files found. Downloading in parallel...
  [READ]  112,650 NDJSON lines downloaded.
  [SHAPE] 112,650 rows × 18 columns
  [DEDUP] 112,650 raw → 21 unique (112,629 duplicates dropped)
  [DELTA] Written to /tmp: /tmp/silver/order_item (2 files)
  [UPLOAD] Uploading 2 Delta files in parallel...
  [UPLOAD] 2 files → gs://[REDACTED]/order_item/
  ✅ SUCCESS  | order_item           |     21 rows |   

duplicates_dropped,elapsed_sec,entity,files_uploaded,raw_count,silver_count,silver_path,status
0,30.7,order,2,99441,99441,gs://silver-processed-project-aaa919f1-4345-401b-860/order/,SUCCESS
112629,25.9,order_item,2,112650,21,gs://silver-processed-project-aaa919f1-4345-401b-860/order_item/,SUCCESS
4446,27.4,payment,2,103886,99440,gs://silver-processed-project-aaa919f1-4345-401b-860/payment/,SUCCESS
814,26.6,review,2,99224,98410,gs://silver-processed-project-aaa919f1-4345-401b-860/review/,SUCCESS
0,26.1,customer,2,99441,99441,gs://silver-processed-project-aaa919f1-4345-401b-860/customer/,SUCCESS
33902,19.3,product,2,66853,32951,gs://silver-processed-project-aaa919f1-4345-401b-860/product/,SUCCESS
0,3.2,product_category,2,71,71,gs://silver-processed-project-aaa919f1-4345-401b-860/product_category/,SUCCESS
0,5.1,seller,2,3095,3095,gs://silver-processed-project-aaa919f1-4345-401b-860/seller/,SUCCESS
981148,186.0,geolocation,2,1000163,19015,gs://silver-processed-project-aaa919f1-4345-401b-860/geolocation/,SUCCESS



✅ All entities processed successfully!


## Cell 7: Verify Silver Output (Spot Check)

Reads the Delta table back from driver `/tmp` using the `deltalake` Python library.

In [0]:
from deltalake import DeltaTable

VERIFY_ENTITY  = "order"
local_verify   = f"/tmp/silver/{VERIFY_ENTITY}"

print(f"Verifying Delta table: '{VERIFY_ENTITY}' at {local_verify}\n")

dt      = DeltaTable(local_verify)
verify_df = dt.to_pandas()

print(f"Schema  : {list(verify_df.dtypes.items())}")
print(f"Rows    : {len(verify_df):,}")
print(f"Columns : {len(verify_df.columns)}")

# Data quality check
null_pk = verify_df["order_id"].isna().sum()
assert null_pk == 0, f"[DQ FAIL] {null_pk} rows with NULL order_id in Silver!"
print(f"\n[DQ PASS] Zero NULL primary keys — Silver data is clean ✅")

display(spark.createDataFrame(verify_df.head(5)))

Verifying Delta table: 'order' at /tmp/silver/order

Schema  : [('order_id', dtype('O')), ('customer_id', dtype('O')), ('order_status', dtype('O')), ('order_purchase_timestamp', datetime64[ns, UTC]), ('order_approved_at', datetime64[ns, UTC]), ('order_delivered_carrier_date', datetime64[ns, UTC]), ('order_delivered_customer_date', datetime64[ns, UTC]), ('order_estimated_delivery_date', datetime64[ns, UTC]), ('event_id', dtype('O')), ('event_type', dtype('O')), ('entity_type', dtype('O')), ('source_system', dtype('O')), ('source_topic', dtype('O')), ('schema_version', dtype('O')), ('ingested_at', dtype('O')), ('ingestion_env', dtype('O')), ('pipeline_run_id', dtype('O')), ('data_layer', dtype('O')), ('checksum', dtype('O')), ('_silver_loaded_at', datetime64[ns, UTC])]
Rows    : 99,441
Columns : 20

[DQ PASS] Zero NULL primary keys — Silver data is clean ✅


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,event_id,event_type,entity_type,source_system,source_topic,schema_version,ingested_at,ingestion_env,pipeline_run_id,data_layer,checksum,_silver_loaded_at
83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27T14:46:43.000Z,2017-08-27T15:04:16.000Z,2017-08-28T20:52:26.000Z,2017-09-21T11:24:17.000Z,2017-09-27T00:00:00.000Z,dba6a0ef-106d-443b-bfd5-a6c20e0717bd,olist.order.created,order,olist-csv-producer,ecommerce.olist.orders.v1,1.0.0,2026-05-17T07:13:11.550494Z,dev,run-ce91cad8,bronze,4dac7eee44e542dc35e46a4332d92d28b4e30a3a969858c4b0559f67dd9d3853,2026-05-17T15:40:32.749Z
66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08T20:57:30.000Z,2018-03-09T11:20:28.000Z,2018-03-09T22:11:59.000Z,2018-03-16T13:08:30.000Z,2018-04-03T00:00:00.000Z,af382de7-e99c-48f7-af56-2d976ed659e6,olist.order.created,order,olist-csv-producer,ecommerce.olist.orders.v1,1.0.0,2026-05-17T07:13:11.550494Z,dev,run-ce91cad8,bronze,7f95c7d1028757976a4a7f076641e6b595508752b0819f7d9641588501d6fdff,2026-05-17T15:40:32.749Z
11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08T21:28:27.000Z,2018-01-08T21:36:21.000Z,2018-01-12T15:35:03.000Z,2018-01-25T23:32:54.000Z,2018-02-15T00:00:00.000Z,e9947f4b-0fdb-476d-afeb-75f3019b77a4,olist.order.created,order,olist-csv-producer,ecommerce.olist.orders.v1,1.0.0,2026-05-17T07:13:11.550494Z,dev,run-ce91cad8,bronze,638662cc0b9c42222b48892e001b9a7487387286ba1f173129985534482dfa0a,2026-05-17T15:40:32.749Z
9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09T09:54:05.000Z,2017-03-09T09:54:05.000Z,2017-03-10T11:18:03.000Z,2017-03-17T15:08:01.000Z,2017-03-28T00:00:00.000Z,ea7da6f3-14eb-42ab-8e3d-091711e76c24,olist.order.created,order,olist-csv-producer,ecommerce.olist.orders.v1,1.0.0,2026-05-17T07:13:11.549495Z,dev,run-ce91cad8,bronze,2fedeecef9113f874a4dbf73537ebcbb26476558e8a8b98226ea82db410b4905,2026-05-17T15:40:32.749Z
880675dff2150932f1601e1c07eadeeb,47cd45a6ac7b9fb16537df2ccffeb5ac,delivered,2017-02-23T09:05:12.000Z,2017-02-23T09:15:11.000Z,2017-03-01T10:22:52.000Z,2017-03-06T11:08:08.000Z,2017-03-22T00:00:00.000Z,abaf5dbd-00b3-4a47-949b-10c30b8417fb,olist.order.created,order,olist-csv-producer,ecommerce.olist.orders.v1,1.0.0,2026-05-17T07:13:11.549495Z,dev,run-ce91cad8,bronze,c1c941dd5bd76e2dc8b9f6899fee4241ad1d171295e27dd64aaa1ee4e9dfd6fb,2026-05-17T15:40:32.749Z
